In [1]:
import pandas as pd
import numpy as np

# ====== CONFIG ======
FILE1 = "risultati_jira-trans-token.csv"
FILE2 = "risultati_kart-trans-token.csv"
OUTPUT_FILE = "risultati_merged_mixed.csv"
RANDOM_STATE = 42

# ====== LOAD ======
print(f"Caricamento {FILE1}...")
df1 = pd.read_csv(FILE1)
print(f"  - Righe: {len(df1)}")
print(f"  - Colonne: {list(df1.columns)}")

print(f"\nCaricamento {FILE2}...")
df2 = pd.read_csv(FILE2)
print(f"  - Righe: {len(df2)}")
print(f"  - Colonne: {list(df2.columns)}")

# ====== AGGIUNTA SOURCE COLUMN (opzionale) ======
# Aggiungi una colonna per identificare la provenienza
df1['source'] = 'jira'
df2['source'] = 'kart'

# ====== MERGE ======
print(f"\nUnione dei dataset...")
df_merged = pd.concat([df1, df2], ignore_index=True)
print(f"  - Totale righe prima del shuffle: {len(df_merged)}")

# ====== SHUFFLE (MIX DELLE RIGHE) ======
print(f"Shuffle delle righe (random_state={RANDOM_STATE})...")
df_merged = df_merged.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

# ====== VERIFICA ======
print(f"\n{'='*60}")
print("STATISTICHE MERGED DATASET")
print(f"{'='*60}")
print(f"Totale righe: {len(df_merged)}")
print(f"Colonne: {len(df_merged.columns)}")
print(f"\nDistribuzione per source:")
print(df_merged['source'].value_counts())

# Verifica colonne chiave
if 'label' in df_merged.columns:
    print(f"\nDistribuzione label:")
    print(df_merged['label'].value_counts())
    
if 'logit_td' in df_merged.columns:
    print(f"\nStatistiche logit_td:")
    print(df_merged['logit_td'].describe())

# Prime righe per verifica mix
print(f"\n{'='*60}")
print("PRIME 10 RIGHE (verifica mix):")
print(f"{'='*60}")
if 'id' in df_merged.columns and 'source' in df_merged.columns:
    print(df_merged[['id', 'source', 'label']].head(10) if 'label' in df_merged.columns 
          else df_merged[['id', 'source']].head(10))
else:
    print(df_merged.head(10))

# ====== SAVE ======
print(f"\nSalvataggio in {OUTPUT_FILE}...")
df_merged.to_csv(OUTPUT_FILE, index=False)
print(f"✓ File salvato con successo!")
print(f"\nRiepilogo:")
print(f"  - Input: {FILE1} ({len(df1)} righe) + {FILE2} ({len(df2)} righe)")
print(f"  - Output: {OUTPUT_FILE} ({len(df_merged)} righe, mescolate)")


Caricamento risultati_jira-trans-token.csv...
  - Righe: 828
  - Colonne: ['numero_frase', 'label', 'orig_text', 'p_td', 'logit_td', 'tokens', 'token_scores']

Caricamento risultati_kart-trans-token.csv...
  - Righe: 20000
  - Colonne: ['id', 'label', 'orig_text', 'p_td', 'logit_td', 'tokens', 'token_scores']

Unione dei dataset...
  - Totale righe prima del shuffle: 20828
Shuffle delle righe (random_state=42)...

STATISTICHE MERGED DATASET
Totale righe: 20828
Colonne: 9

Distribuzione per source:
source
kart    20000
jira      828
Name: count, dtype: int64

Distribuzione label:
label
0    10453
1    10375
Name: count, dtype: int64

Statistiche logit_td:
count    20828.000000
mean        -0.179584
std          4.125192
min         -4.432512
25%         -4.386106
50%         -1.579602
75%          4.225060
max          4.330991
Name: logit_td, dtype: float64

PRIME 10 RIGHE (verifica mix):
             id source  label
0  2.119501e+10   kart      0
1  3.065498e+10   kart      1
2  2.003